# TODO: REMOVE DATAFRAME CASTING HERE AND IN PREVIOUS NOTEBOOKS

# General Preprocessing
While some preprocessing steps are unique to supervised machine learning models, others apply to all.

Such general preprocessing happens in two steps.

The first step is to remove whole articles that do not contribute to the classification task by:

- Removing articles which appear more than once in one dataset (duplicates)
- Removing non-English abstracts
- Removing long abstracts which contain more than 489 words
- Removing articles without abstracts

The second step is to clean the texts of the remaining articles from noise, namely:

- Removing HTML
- Removing non-character symbols, such as digits and special characters

## Custom Modules
The ``src`` directory houses custom modules with functions that will be reused throughout the project.

To be able to import these moduls, we begin with programmatically adding the project's root directory to ``sys.path``.

After adding the root to ``sys.path``, we can import the ``data`` and ``util`` modules:

In [1]:
import os, sys

# recursively search for the root directory containing a specific file
def find_root_dir(search_for='.gitignore'):

    current_dir = os.getcwd()

    while True:
        if os.path.exists(os.path.join(current_dir, search_for)):
            return current_dir
        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir:
            raise FileNotFoundError(f"Could not find '{search_for}' in any parent directory.")
        current_dir = parent_dir


# save the root directory to a variable
root_dir = find_root_dir()
print(f"Root directory found: {root_dir}")

# add the root directory to the system path
sys.path.append(root_dir)
if root_dir in sys.path:
    print(f"Root directory added to system path.")


# import custom modules
from src import data

Root directory found: c:\dev\automated_title_abstract_screening
Root directory added to system path.


# Preprocessing
## Preparation

### Datasets

In [2]:
from src import data
import polars as pl

# Define the directory where the data is stored
directory = '../../../data/datasets/03_pubmed'

# Output directory
output_directory = '../../../data/datasets/04_preprocessed'

# Load the data
# TODO: Check whether have to change with_index to True
datasets = data.dict_from_directory(directory, type='polars', with_index=True)

## Removal of Abstracts and Articles
Remove non-English abstracts and abstracts with more than 489 words.

Afterwards, remove articles without an abstract, as those do not contain sufficient information to decide on inclusion or exclusion in title/abstract-screening.

### Duplicates

In [3]:
# columns with duplicates known from manual inspection of the data
# within the original study there were also duplicates in pancreatic surgery
known_duplicates = {
    'animal_depression': ['doi', 'openalex_id', 'pubmed_id'],
}

# remove duplicates for all columns of all datasets with known duplicates
# originally, pancreatic surgery was also included, but it was removed
# since the data is not publicly available
for subject, dataset in datasets.items():
    if subject in known_duplicates.keys():
        for column in known_duplicates[subject]:

            # save nulls as they are lost to .unique()
            null = datasets[subject].filter(
                pl.col(column).is_null() 
            )

            # drop duplicate values - also drops null values
            unique = datasets[subject].filter(
                pl.col(column).is_unique()
            )

            # combine the unique and null values
            filtered = unique.vstack(null)

            # override the dataset with the filtered dataset
            datasets[subject] = filtered 

### Non-English Abstracts
In some instances ``fast-langdetect`` predicted the wrong language.

Hard-code the indices of undetected English abstracts and such that are actually within another language. 

In [4]:
actually_english = {
    'adhd': [521],
    'animal_depression': [16, 743, 1528],
}

not_english = {
    'adhd': [667,759,785,801],
    'animal_depression': [266, 421, 675, 848, 968, 1162, 1250, 1521, 1644, 1919, 1947],
}

Replace the language code for the actual English abstracts with 'en'. Then, remove the abstracts of articles with a language code other than 'en'.

In [5]:
# iterate over the datasets and correct the language labels
for subject, dataset in datasets.items():


    if subject in actually_english.keys():
        dataset =  dataset.with_columns(
            pl.when(
                pl.col('index').is_in(actually_english[subject]),
            ).then(
                pl.lit('en')
            ).otherwise(
                pl.col('language_abstract')).alias('language_abstract')
        )

        # delete non-English abstracts to later remove all articles 
        # without an abstract alltogether
        dataset = dataset.with_columns(
            pl.when(
                pl.col('index').is_in(not_english[subject]),
            ).then(pl.lit(None))
            .otherwise(pl.col('abstract'))
            .alias('abstract')
        )

        datasets[subject] = dataset

Verify that the abstracts of articles with a ``language_abstract`` label other than ``en`` have been removed:

In [6]:
datasets['adhd'].vstack(
    datasets['animal_depression']
).filter(
    pl.col('language_abstract') != 'en'
).select(['index', 'abstract', 'language_abstract'])

index,abstract,language_abstract
u32,str,str
667,null,"""ceb"""
759,null,"""de"""
785,null,"""pt"""
801,null,"""de"""
266,null,"""de"""
…,…,…
1250,null,"""de"""
1521,null,"""es"""
1644,null,"""de"""


### Long Abstracts
Remove abstracts that have more words than the limit of 489 words:

In [7]:
WORD_LIMIT = 489

# iterate over the datasets and remove abstracts that are above the word limit
for subject, dataset in datasets.items():


    # list of indices of abstracts that are above the word limit
    idx_long_abstracts = dataset.filter(
        pl.col('abstract_word_count') > WORD_LIMIT
    ).select('index').to_series().to_list()

    # remove abstracts that are above the word limit
    dataset = dataset.with_columns(
            pl.when(
                pl.col('index').is_in(idx_long_abstracts),
            ).then(pl.lit(None))
            .otherwise(pl.col('abstract'))
            .alias('abstract')
        )
    
    datasets[subject] = dataset

## Missing Abstracts
Now among articles without an abstract are  those that did not have an abstract to begin with or those whose abstract has been removed for being non-English or too long.

Remove such articles without an abstract as they do not contain sufficient information to decide on inclusion or exclusion:

In [8]:
# to document how many abstracts are removed
documentation = pl.DataFrame(
    {
        'dataset': datasets.keys(),
    }
)

# lengths of the datasets before removing articles with empty abstracts
lengths_before = [len(dataset) for dataset in datasets.values()]

# add the initial lengths of the datasets
documentation = documentation.with_columns(
    pl.Series("length_before", lengths_before)
)

In [9]:
# remove articles with empty abstracts
for subject, dataset in datasets.items():
    
    datasets[subject] = dataset.filter(
        pl.col('abstract').is_not_null()
    )

Validate, how many articles were removed due to missing abstracts:

In [10]:
# lengths of the datasets after removing articles with empty abstracts
lengths_after = [len(dataset) for dataset in datasets.values()]

# how many articles were removed, absolutely and relative to the total
documentation.with_columns(
    pl.Series("length_after", lengths_after)
).with_columns(
    (
        pl.col('length_before') - pl.col('length_after')
    ).alias('abstracts_removed')
).with_columns(
    (
        (pl.col('abstracts_removed') / pl.col('length_before')) * 100
    ).alias('percentage_removed')
).sort(by='percentage_removed')

dataset,length_before,length_after,abstracts_removed,percentage_removed
str,i64,i64,i64,f64
"""adhd""",851,798,53,6.227967
"""atypical_antipsychotics""",1120,1049,71,6.339286
"""calcium_channel_blockers""",1218,1129,89,7.307061
"""oral_hypoglycemics""",503,458,45,8.946322
"""animal_depression""",1989,1691,298,14.982403


## Noise Removal
### HTML
HTML tags cause noise within the texts:

In [11]:
before = datasets['adhd'].to_pandas().iloc[[456, 565]].abstract.values

Define a function to automatically remove HTML from the text

In [12]:
from bs4 import BeautifulSoup

def remove_html(text: str)-> str:
    """Remove html tags from a string
    
    Args:
    text: str: a string containing html tags

    Returns:
    str: a string without html tags
    """
    return BeautifulSoup(text, 'html.parser').get_text()

Remove HTML from all datasets:

In [13]:
import polars as pl

for subject, dataset in datasets.items():
    datasets[subject] = dataset.with_columns([
        pl.col('title').map_elements(remove_html, return_dtype=pl.String),
        pl.col('abstract').map_elements(remove_html, return_dtype=pl.String)
    ])

Verify that HTML was indeed removed:

In [14]:
after = datasets['adhd'].to_pandas().iloc[[456, 565]].abstract.values

print('With HTML:', end='\n\n')
[print(abstract[:100], end='\n') for abstract in before];
print('\n\nWithout HTML:', end='\n\n')
[print(abstract[:100], end='\n') for abstract in after];

With HTML:

Patients with myotonic dystrophy frequently suffer from excess daytime sleepiness, which can be a si
In a randomized, double-blind study in children undergoing elective orthopaedic surgery, we have ass


Without HTML:

Patients with myotonic dystrophy frequently suffer from excess daytime sleepiness, which can be a si
In a randomized, double-blind study in children undergoing elective orthopaedic surgery, we have ass


# Characters Only
Remove all digits and special characters by regular expressions:

In [15]:
import re

def remove_special_characters(text: str) -> str:
    """Remove special characters from a string
    
    Args:
    text: str: a string containing special characters

    Returns:
    str: a string without special characters
    """
    # remove newlines and carriage returns
    text = text.replace('\n', ' ').replace('\r', '')
    
    # matches characters only
    pattern = r'[^a-zA-Z\s]+'

    # apply the pattern to clean the string
    clean_string = re.sub(pattern, '', text)

    # ensure that there are no multiple spaces
    clean_string = ' '.join(clean_string.split())

    return clean_string

Validate that the function works as expected:

In [16]:
test = 'This,  i2s A  t3st!\n\r4nd  1t  w0rks.'
remove_special_characters(test)

'This is A tst nd t wrks'

Apply the function to all titles and abstracts to keep characters only:

In [17]:
for subject, dataset in datasets.items():
    datasets[subject] = dataset.with_columns([
        pl.col('title').map_elements(
            remove_special_characters,
            return_dtype=pl.String
        ),
        pl.col('abstract').map_elements(
            remove_special_characters,
            return_dtype=pl.String
        )
    ])

# Export
Export the preprocessed data for classification:

In [ ]:
for subject, dataset in datasets.items():
    dataset.write_csv(f'{output_directory}/{subject}_preprocessed.csv')